# GBM From Ito's lemma
Start with the GBM SDE mentioned earlier:
$$dS_t = μS_tdt + σS_tdW_t \quad (1)$$

Now, let $X_t = ln(S_t)$ and let $S_0$ be an arbitrary positive and non-zero initial value. By Ito's Lemma, for a twice continously differentiable function $f(S_t)$, the differential $df$ is:
$$df(S_t) = f'(S_t)dS_t + \frac{1}{2}f''(S_t)(dS_t)^2 \quad (2)$$

Applying (2) to $X_t$, and knowing $(dS_t)^2 = σ^2S^2_tdt$ (the quadratic variation of $S_t$ as $dt\to0$, the computation is out of scope for this section so we will take the result for granted),
$$dX_t = \frac{dX_t}{dS_t}dS_t + \frac{1}{2}\frac{d^2X_t}{(dS_t)^2}(dS_t)^2$$

$$= \frac{1}{S_t}dS_t + \frac{-1}{2S^2_t}(σ^2S^2_tdt)$$

$$= \frac{1}{S_t}(μS_tdt + σS_tdW_t) - \frac{1}{2}σ^2dt$$

$$=μdt + σdW_t - \frac{1}{2}σ^2dt$$

$$=dt(μ - \frac{1}{2}σ^2) + σdW_t$$

Now, integrate both sides, plugging in $X_t = ln(S_t)$,

$$\int{d(ln(S_t))} = (μ - \frac{1}{2}σ^2)\int{dt} + σ\int{dW_t}$$

$$ln(S_t) = (μ - \frac{1}{2}σ^2)t + σW_t + C$$

Let $C = ln(S_0)$ and plug back in,

$$ln(S_t) = (μ - \frac{1}{2}σ^2)t + σW_t + ln(S_0)$$

$$ln(\frac{S_t}{S_0}) = (μ - \frac{1}{2}σ^2)t + σW_t$$

$$S_t = S_0\exp((μ - \frac{σ^2}{2})t + σW_t)\quad (3)$$

This is exactly Equation (2) in notebook 1, and thus we have achieved our goal of deriving GBM from its SDE using Ito's Lemma.

In [ ]:
import numpy as np
import pandas as pd
import numpy.typing as npt

from data.data import get_multiple_stocks_data

In [ ]:
class GBMRiskEngine:
    ROLLING_PERIOD = 60
    DAILY = 252
    N_PATHS = 10_000

    def simulate_euler_maruyama(
        self, 
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
      N = int(np.floor(N))
      z = np.random.standard_normal((n_paths, N))
      delta_t = T / N
      paths = np.zeros((n_paths, N + 1))
      paths[:, 0] = S0
      paths[:, 1:] = S0 * np.cumprod(1 + mu_annual * delta_t + sigma_annual * np.sqrt(delta_t) * z, axis=1)
      return paths

    def simulate_gbm(
        self,
        S0: float,
        mu_annual: float,
        sigma_annual: float,
        T: float,
        N: int,
        n_paths: int
    ) -> npt.NDArray[np.float64]:
        N = int(N)
        dt = T / N
        z = np.random.standard_normal((n_paths, N))

        log_increments = (
            (mu_annual - 0.5 * sigma_annual**2) * dt
            + sigma_annual * np.sqrt(dt) * z
        )

        paths = np.zeros((n_paths, N + 1))
        paths[:, 0] = S0
        paths[:, 1:] = S0 * np.exp(np.cumsum(log_increments, axis=1))
        return paths

    def var_es(
        self,
        paths: npt.NDArray[np.float64],
        time_horizon: int,
        alpha: float
    ) -> tuple[float, float]:
        if time_horizon < 1 or time_horizon >= paths.shape[1]:
            raise ValueError("time_horizon must be between 1 and number of simulated steps")

        P0 = paths[0, 0]
        losses = P0 - paths[:, time_horizon]

        var = np.percentile(losses, (1 - alpha) * 100)
        es = losses[losses >= var].mean()

        return float(var), float(es)

In [ ]:
risk_eng = GBMRiskEngine()

tickers = ["SPY", "GLD", "NVDA", "GOOGL", "BTC-USD"]
data = get_multiple_stocks_data(tickers, "2016-06-01", "2026-06-01", "1d")

prices = {}
paths_by_ticker = {}
var_results = {}
es_results = {}

for ticker in tickers:
    curr_data = data[ticker]
    prices[ticker] = curr_data["Close"]

    lr = np.diff(np.log(prices[ticker].values))

    mu_annual = lr.mean() * risk_eng.DAILY
    sigma_annual = lr.rolling(risk_eng.ROLLING_PERIOD).std().dropna().iloc[-1] * np.sqrt(risk_eng.DAILY)

    paths = risk_eng.simulate_gbm(
        S0=float(prices[ticker].iloc[-1]),
        mu_annual=float(mu_annual),
        sigma_annual=float(sigma_annual),
        T=1.0,
        N=risk_eng.DAILY,
        n_paths=risk_eng.N_PATHS
    )
    paths_by_ticker[ticker] = paths

    v95_1, e95_1 = risk_eng.var_es(paths, 1, 0.05)
    v99_1, e99_1 = risk_eng.var_es(paths, 1, 0.01)
    v95_10, e95_10 = risk_eng.var_es(paths, 10, 0.05)
    v99_10, e99_10 = risk_eng.var_es(paths, 10, 0.01)

    var_results[ticker] = {
        "1d": {"var_95": v95_1, "var_99": v99_1},
        "10d": {"var_95": v95_10, "var_99": v99_10},
    }

    es_results[ticker] = {
        "1d": {"es_95": e95_1, "es_99": e99_1},
        "10d": {"es_95": e95_10, "es_99": e99_10},
    }

In [ ]:
print("Var Results:")
for ticker, results in var_results.items():
    print(f"{ticker}:")
    for time_horizon, metrics in results.items():
        print(f"  {time_horizon}d: VAR 95% = {metrics['var_95']:.2f}, VAR 99% = {metrics['var_99']:.2f}")

print("\nES Results:")
for ticker, results in es_results.items():
    print(f"{ticker}:")
    for time_horizon, metrics in results.items():
        print(f"  {time_horizon}d: ES 95% = {metrics['es_95']:.2f}, ES 99% = {metrics['es_99']:.2f}")